# AAPlot Animal Locomotion – Speed Analysis & Batch Statistics

This notebook merges the per‑file **speed/trajectory plotting** workflow from
`speedAnalysis&plot[single]` with the **batch‑processing statistics** pipeline
from `analysis[batch]`.

**Functionality:**
1. search a folder for Spike2 `.txt`/`.csv` exports
2. for each file: read, clean, compute speed, smooth and plot, save outputs
3. aggregate summary statistics across files (mean speeds, distances, etc.)
4. export both individual cleaned data and combined summary CSV

Run in the `PLOT` conda environment (Python 3.12+ with pandas/numpy/...) as before.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import os
import glob


# --------------------------parameter input-----------------------------
# set folder path here (adjust as needed)
folder_path = r'D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST'

# manually input for event times (in seconds) by mouse number (file prefix)
manual_event_times = {
    'C89': 2750,
    'C90': 2590,
    'C91': 3000,
    'C92': 3600,
    'C93': 3700,
    'C94': 3800,
    'C95': 3600,
    'C96': 3600
}


# main batch processing code
# --------------------- no need to be modified below --------------------
# collect files
data_files = []
for ext in ['.csv']:
    data_files.extend(glob.glob(os.path.join(folder_path, f'*{ext}')))

if not data_files:
    raise ValueError(f"No .csv files found in {folder_path}")

print(f"Found {len(data_files)} files")
for f in data_files:
    print("-", os.path.basename(f))


# output directory for all results
output_dir = os.path.join(folder_path, 'combined_analysis')
os.makedirs(output_dir, exist_ok=True)

all_results = []
# dictionary mapping file-prefix (chars before first underscore) -> event time (seconds)


for fp in data_files:
    base = os.path.splitext(os.path.basename(fp))[0]
    prefix = base.split('_', 1)[0]
    override = manual_event_times.get(prefix)
    if override is not None:
        print(f"using manual override for prefix {prefix}: {override}")
    try:
        res = process_file(fp, output_dir, event_time_override=override)
        all_results.append(res)
    except Exception as e:
        print(f"error processing {os.path.basename(fp)}: {e}")

# produce dataframe and save summary
if all_results:
    summary_df = pd.DataFrame(all_results)
    summary_file = os.path.join(output_dir, 'batch_summary.csv')
    summary_df.to_csv(summary_file, index=False)
    print("Batch summary saved to", summary_file)
    print(summary_df.head())
else:
    print("No results produced.")




# ----------------------Functions ---------------------------
# read files under the folder
def read_data_file(filepath):
    """Simply load the CSV and return the DataFrame."""
    return pd.read_csv(filepath)


def process_file(filepath, output_dir, event_time_override=None):
    """Clean, analyze and plot a single file; return summary dict.

    Parameters
    ----------
    filepath : str
        Path to input file.
    output_dir : str
        Directory where plots and cleaned CSVs will be written.
    event_time_override : float or None
        If provided, this value (seconds) will be used as the event time
        instead of trying to infer it from a marker column. Useful when the
        file contains no markers.
    """
    print(f"\n--- processing {filepath} ---")
    df = read_data_file(filepath)
    base = os.path.splitext(os.path.basename(filepath))[0]
    print(f"--- base name {base} ---")
    print(f"columns: {list(df.columns)}")

    # convert eztrack style if present
    if 'Frame' in df.columns and 'Distance_cm' in df.columns:
        time_secs = df['Frame'] / 30
        speed = df['Distance_cm'] * 30
        dfc = pd.DataFrame({'Time': time_secs, 'Speed': speed})
        dfc['Marker'] = 0
    else:
        dfc = df.copy()
        if 'Time' not in dfc.columns or 'Speed' not in dfc.columns:
            raise ValueError('input file missing Time/Speed columns')
        dfc['Time'] = pd.to_numeric(dfc['Time'], errors='coerce')
        dfc['Speed'] = pd.to_numeric(dfc['Speed'], errors='coerce')
        if 'Marker' not in dfc.columns:
            dfc['Marker'] = 0

    # data cleaning, delete impossible values and replace the NAN
    dfc['Speed'] = dfc['Speed'].clip(upper=100)
    dfc['Speed'].interpolate(inplace=True)

    # determine event time
    if event_time_override is not None: # user defined
        event_time = event_time_override
        print(f"using manual event time {event_time} s")
    else:
        ev = dfc[dfc['Marker'] == 1] # If there is a marker column, named 'Marker', 1 for event, 0 otherwise
        if len(ev):
            event_time = ev.iloc[0]['Time']
            print(f"inferred event time {event_time:.2f} s from marker")
        else:
            event_time = 3600  # default to 1h session if no marker or override
            print(f"no marker; using midpoint event time {event_time:.2f} s")
    dfc['Time_hours'] = (dfc['Time'] - event_time) / 3600

    # stats 
    windows = {
        'pre_1h': (-1, 0),
        'post_0_to_1h': (0, 1),
        'post_1_to_2h': (1, 2),
        'post_2_to_3h': (2, 3),
    }
    speed_stats = {}
    dist_stats = {}
    for name, (start, end) in windows.items():
        mask = (dfc['Time_hours'] >= start) & (dfc['Time_hours'] < end)
        w = dfc[mask]
        if len(w):
            speed_stats[name] = {'mean': w['Speed'].mean(),
                                  'std': w['Speed'].std(),
                                  'n': len(w)}
            speeds = w['Speed'] * 3600
            dt = np.diff(w['Time_hours'])
            dist = np.sum(speeds[:-1] * dt)
            dist_stats[name] = dist / 100
        else:
            speed_stats[name] = {'mean': np.nan, 'std': np.nan, 'n': 0}
            dist_stats[name] = np.nan

    # plot
    smooth = savgol_filter(dfc['Speed'], window_length=min(31,len(dfc['Speed']))//2*2+1, polyorder=2)
    smooth = np.clip(smooth,0,None)
    fig, ax = plt.subplots(figsize=(6,2.5))
    ax.plot(dfc['Time'], smooth, color='k', linewidth=1)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Speed (cm/s)')
    ax.set_title(base)
    plt.tight_layout()
    plotfile = os.path.join(output_dir, f"{base}_speed.svg")
    fig.savefig(plotfile, dpi=300)
    plt.close(fig)

    # save cleaned data
    datafile = os.path.join(output_dir, f"{base}_processed.csv")
    dfc.to_csv(datafile, index=False)

    # build summary dictionary and return
    summary = {'file': base}
    summary.update({f"speed_{k}": v['mean'] for k, v in speed_stats.items()})
    summary.update({f"dist_{k}": v for k, v in dist_stats.items()})

    return summary

In [ ]:
# set folder path here (adjust as needed)
folder_path = r'D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST'

# collect files
data_files = []
for ext in ['.csv']:
    data_files.extend(glob.glob(os.path.join(folder_path, f'*{ext}')))

if not data_files:
    raise ValueError(f"No .csv files found in {folder_path}")

print(f"Found {len(data_files)} files")
for f in data_files:
    print("-", os.path.basename(f))


# output directory for all results
output_dir = os.path.join(folder_path, 'combined_analysis')
os.makedirs(output_dir, exist_ok=True)

all_results = []
# dictionary mapping file-prefix (chars before first underscore) -> event time (seconds)
manual_event_times = {
    'C89': 2750,
    'C90': 2590,
    'C91': 3000,
    'C92': 3600,
    'C93': 3700,
    'C94': 3800,
    'C95': 3600,
    'C96': 3600
}

for fp in data_files:
    base = os.path.splitext(os.path.basename(fp))[0]
    prefix = base.split('_', 1)[0]
    override = manual_event_times.get(prefix)
    if override is not None:
        print(f"using manual override for prefix {prefix}: {override}")
    try:
        res = process_file(fp, output_dir, event_time_override=override)
        all_results.append(res)
    except Exception as e:
        print(f"error processing {os.path.basename(fp)}: {e}")

# produce dataframe and save summary
if all_results:
    summary_df = pd.DataFrame(all_results)
    summary_file = os.path.join(output_dir, 'batch_summary.csv')
    summary_df.to_csv(summary_file, index=False)
    print("Batch summary saved to", summary_file)
    print(summary_df.head())
else:
    print("No results produced.")

Found 8 files
- C89_LocationOutput.csv
- C90_LocationOutput.csv
- C91_LocationOutput.csv
- C92_LocationOutput.csv
- C93_LocationOutput.csv
- C94_LocationOutput.csv
- C95_LocationOutput.csv
- C96_LocationOutput.csv
using manual override for prefix C89: 2750

--- processing D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST\C89_LocationOutput.csv ---
--- base name C89_LocationOutput ---
columns: ['File', 'Location_Thresh', 'Use_Window', 'Window_Weight', 'Window_Size', 'Start_Frame', 'Frame', 'X', 'Y', 'a', 'ROI_coordinates', 'ROI_location', 'ROI_transition', 'Distance_px', 'Distance_cm']
using manual event time 2750 s


C:\Users\RuyiCai\AppData\Local\Temp\ipykernel_40840\3323544606.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfc['Speed'].interpolate(inplace=True)


using manual override for prefix C90: 2590

--- processing D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST\C90_LocationOutput.csv ---
--- base name C90_LocationOutput ---
columns: ['File', 'Location_Thresh', 'Use_Window', 'Window_Weight', 'Window_Size', 'Start_Frame', 'Frame', 'X', 'Y', 'a', 'ROI_coordinates', 'ROI_location', 'ROI_transition', 'Distance_px', 'Distance_cm']
using manual event time 2590 s


C:\Users\RuyiCai\AppData\Local\Temp\ipykernel_40840\3323544606.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfc['Speed'].interpolate(inplace=True)


using manual override for prefix C91: 3000

--- processing D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST\C91_LocationOutput.csv ---
--- base name C91_LocationOutput ---
columns: ['File', 'Location_Thresh', 'Use_Window', 'Window_Weight', 'Window_Size', 'Start_Frame', 'Frame', 'X', 'Y', 'a', 'ROI_coordinates', 'ROI_location', 'ROI_transition', 'Distance_px', 'Distance_cm']
using manual event time 3000 s


...

using manual override for prefix C95: 3600

--- processing D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST\C95_LocationOutput.csv ---
--- base name C95_LocationOutput ---
columns: ['File', 'Location_Thresh', 'Use_Window', 'Window_Weight', 'Window_Size', 'Start_Frame', 'Frame', 'X', 'Y', 'a', 'ROI_coordinates', 'ROI_location', 'ROI_transition', 'Distance_px', 'Distance_cm']
using manual event time 3600 s


C:\Users\RuyiCai\AppData\Local\Temp\ipykernel_40840\3323544606.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfc['Speed'].interpolate(inplace=True)


using manual override for prefix C96: 3600

--- processing D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST\C96_LocationOutput.csv ---
--- base name C96_LocationOutput ---
columns: ['File', 'Location_Thresh', 'Use_Window', 'Window_Weight', 'Window_Size', 'Start_Frame', 'Frame', 'X', 'Y', 'a', 'ROI_coordinates', 'ROI_location', 'ROI_transition', 'Distance_px', 'Distance_cm']
using manual event time 3600 s


C:\Users\RuyiCai\AppData\Local\Temp\ipykernel_40840\3323544606.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfc['Speed'].interpolate(inplace=True)


Batch summary saved to D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST\combined_analysis\batch_summary.csv
                 file  speed_pre_1h  speed_post_0_to_1h  speed_post_1_to_2h  \
0  C89_LocationOutput      9.761536           12.172044            7.056432   
1  C90_LocationOutput      8.282189           10.541824            6.929876   
2  C91_LocationOutput     21.891242           20.803296           17.977207   
3  C92_LocationOutput     11.124261           10.661762                 NaN   
4  C93_LocationOutput     14.970443            9.998114            7.236001   

   speed_post_2_to_3h  dist_pre_1h  dist_post_0_to_1h  dist_post_1_to_2h  \
0            5.950294   268.419166         438.193359         254.022550   
1                 NaN   214.506861         379.501889         206.604854   
2           20.464590   656.737145         748.918006         647.177496   
3                 NaN   400.459181         219.390075                NaN   
4            9.221109   538.933433  

In [37]:
# test returning keys for first file
if data_files:
    debug_res = process_file(data_files[0], output_dir)
    print('returned keys:', debug_res.keys())


--- processing D:\Temp\DrugIntake behavior Analysis\Cocaine\TEST\C89_LocationOutput.csv ---
--- base name C89_LocationOutput ---
columns: ['File', 'Location_Thresh', 'Use_Window', 'Window_Weight', 'Window_Size', 'Start_Frame', 'Frame', 'X', 'Y', 'a', 'ROI_coordinates', 'ROI_location', 'ROI_transition', 'Distance_px', 'Distance_cm']
no marker; using midpoint event time 3600.00 s


C:\Users\RuyiCai\AppData\Local\Temp\ipykernel_40840\3323544606.py:54: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dfc['Speed'].interpolate(inplace=True)


returned keys: dict_keys(['file', 'speed_pre_1h', 'speed_post_0_to_1h', 'speed_post_1_to_2h', 'speed_post_2_to_3h', 'dist_pre_1h', 'dist_post_0_to_1h', 'dist_post_1_to_2h', 'dist_post_2_to_3h'])
